In [0]:
from pyspark.sql.functions import current_timestamp

# Configuration
source_path = "/Volumes/data_modelling_demo/source/demo_volume/scd2/input files/"
target_table = "data_modelling_demo.bronze.scd2"
checkpoint_path = "/Volumes/data_modelling_demo/source/demo_volume/scd2/checkpoints/"
schema_location = "/Volumes/data_modelling_demo/source/demo_volume/scd2/schema/"

# Read CSV files using Auto Loader in batch mode
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)  # Auto Loader handles schema inference
    .option("header", "true")
    .option("inferSchema", "true")
    .load(source_path)
    .withColumn("insert_timestamp", current_timestamp())
)

# Write to bronze table using batch mode (trigger availableNow)
query = (df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

# Wait for the batch processing to complete
query.awaitTermination()

print(f"Data successfully loaded to {target_table}")



Data successfully loaded to data_modelling_demo.bronze.scd2


In [0]:
%sql
-- Validation Query 1: Check total row count
SELECT COUNT(*) AS total_rows
FROM data_modelling_demo.bronze.scd2;

total_rows
8


In [0]:
%sql
-- Validation Query 2: Preview sample records with insert_timestamp
SELECT *
FROM data_modelling_demo.bronze.scd2
LIMIT 10;

customer_id,first_name,email,city,registration_date,last_updated_ts,_rescued_data,insert_timestamp
C001,john,JOHN.SMITH@gmail.com,new york,10-Jan-2024,2024-01-10 10:15:00,null,2026-06-24T12:09:30.082Z
C002,alice,alice.johnson@gmail.com,boston,11-Jan-2024,2024-01-11 11:20:00,null,2026-06-24T12:09:30.082Z
C003,michael,michael.brown@gmail.com,chicago,12-Jan-2024,2024-01-12 09:10:00,null,2026-06-24T12:09:30.082Z
C004,sarah,sarah.davis@gmail.com,seattle,13-Jan-2024,2024-01-13 15:45:00,null,2026-06-24T12:09:30.082Z
C005,david,david.wilson@gmail.com,dallas,14-Jan-2024,2024-01-14 16:00:00,null,2026-06-24T12:09:30.082Z
C001,john,john.smith@gmail.com,chicago,10-Jan-2024,2024-02-01 09:30:00,null,2026-06-25T12:22:41.805Z
C002,alice,alice123@gmail.com,boston,11-Jan-2024,2024-02-01 10:45:00,null,2026-06-25T12:22:41.805Z
C006,emma,emma.thomas@gmail.com,san francisco,01-Feb-2024,2024-02-01 11:15:00,null,2026-06-25T12:22:41.805Z


In [0]:
%sql
-- Validation Query 3: Verify table schema and column types
DESCRIBE data_modelling_demo.bronze.scd2;

col_name,data_type,comment
customer_id,string,null
first_name,string,null
email,string,null
city,string,null
registration_date,string,null
last_updated_ts,string,null
_rescued_data,string,null
insert_timestamp,timestamp,null


In [0]:
%sql
-- Validation Query 4: Check insert_timestamp column
-- Verify all records have timestamp and check the load date range
SELECT 
    COUNT(*) AS total_records,
    COUNT(insert_timestamp) AS records_with_timestamp,
    MIN(insert_timestamp) AS earliest_load_time,
    MAX(insert_timestamp) AS latest_load_time,
    COUNT(*) - COUNT(insert_timestamp) AS null_timestamps
FROM data_modelling_demo.bronze.scd2;

total_records,records_with_timestamp,earliest_load_time,latest_load_time,null_timestamps
8,8,2026-06-24T12:09:30.082Z,2026-06-25T12:22:41.805Z,0


In [0]:
%sql
-- Validation Query 5: Check Delta table history and metadata
-- This shows all operations performed on the table
DESCRIBE HISTORY data_modelling_demo.bronze.scd2 LIMIT 5;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-06-25T12:22:46.000Z,75019635161112,brnikumbh7@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 80cb7259-df7e-4c76-b748-e4421923497b, epochId -> 1, statsOnLoad -> true)",null,List(978838486285116),53313439-1103-4c27-87ba-939a0a045714,0625-121135-tc43ldct-v2n,1,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 3, numOutputBytes -> 2728, numAddedFiles -> 1)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
1,2026-06-24T12:09:34.000Z,75019635161112,brnikumbh7@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 80cb7259-df7e-4c76-b748-e4421923497b, epochId -> 0, statsOnLoad -> true)",null,List(978838486285116),05f10b21-42c3-4a86-8703-992c82aaf185,0624-115728-zie3d2sj-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 5, numOutputBytes -> 2812, numAddedFiles -> 1)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
0,2026-06-24T12:09:25.000Z,75019635161112,brnikumbh7@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-4677465a-ff4c-4c5f-9317-2d8452abd675"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-56c1c238-e910-44c0-853c-e071e5a19655""}, statsOnLoad -> false)",null,List(978838486285116),05f10b21-42c3-4a86-8703-992c82aaf185,0624-115728-zie3d2sj-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13


In [0]:
%sql
-- Validation Query 6: Data quality check
-- Count records by load batch (grouped by insert_timestamp date)
SELECT 
    DATE(insert_timestamp) AS load_date,
    COUNT(*) AS records_loaded
FROM data_modelling_demo.bronze.scd2
GROUP BY DATE(insert_timestamp)
ORDER BY load_date DESC;

load_date,records_loaded
2026-06-25,3
2026-06-24,5
